# Discrete and hybrid systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/advanced/01_discrete_and_hybrid.ipynb)

Tick laws and laws on a clock, multi-rate loops on a continuous plant, the two runners, and discrete-time analysis in $z$.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Tick laws and laws on a clock

A discrete law comes in two forms:

- a **tick law** counts ticks $k$ and knows no seconds: a `StepSystem`, $x_{k+1} = \mathrm{step}(x_k, u_k, k)$, or a `StepDiagramSystem` of them;
- a **law on a clock** ticks every $\Delta t$ seconds: `law % dt`, a `SampledSystem` (or a `SampledDiagramSystem` for a diagram).

The clock is applied from outside, never written into `step`. Here is a first-order
low-pass filter as a tick law, $x_{k+1} = \alpha x_k + (1 - \alpha) u_k$, rolled out for
five ticks on a unit input.

In [ ]:
import numpy as np

from minilink import StepSystem


class LowPass(StepSystem):
    def __init__(self, alpha=0.6):
        super().__init__(1, input_dim=1, output_dim=1)
        self.params = {"alpha": alpha}

    def step(self, x, u, k=0, params=None):
        params = self.params if params is None else params
        alpha = params["alpha"]

        x_next = alpha * x + (1.0 - alpha) * u

        return x_next


low_pass = LowPass()
ticks = low_pass.compute_rollout(n_steps=5, u=np.array([1.0]))
ticks.t, ticks.x

`low_pass % 0.01` is the same law on a 100 Hz clock: the values are identical, the time
axis is now in seconds. A diagram of tick laws goes on a clock as a whole: each block is
clocked, the wiring is kept.

In [ ]:
clocked = low_pass % 0.01
on_clock = clocked.compute_rollout(n_steps=5, u=np.array([1.0]))
print(on_clock.t, np.array_equal(on_clock.x, ticks.x))

two_stages = LowPass() >> LowPass()
print(two_stages)
print(two_stages % 0.01)

A continuous law goes on a clock the same way. `law % dt` is then its Euler difference
equation, $z_{k+1} = z_k + \Delta t\, f(z_k, e_k, t_k)$: the controller you would code on a
microcontroller, not the integrated plant (that is `discretize`).

In [ ]:
from minilink.blocks import LowPassFilter

analog_filter = LowPassFilter(cutoff_hz=3.0)
digital_filter = analog_filter % 0.01
digital_filter.kind, digital_filter.sample_period

## 2. A multi-rate computer on a continuous plant

Laws on different clocks share one loop. The computer below filters the measurement at
100 Hz and runs a proportional law at 10 Hz on a damped mass-spring plant. `closed_loop`
places the filter in the feedback path; the loop ticks on the fastest clock, and each law
fires on its own multiple of it. Between ticks the held command drives the plant in
continuous time.

In [ ]:
from minilink import P, SingleMass, Step, closed_loop

plant = SingleMass()
plant.params["b"] = 1.0

loop = closed_loop(
    P(5.0, ports="reference") % 0.1,
    plant,
    reference=Step(np.zeros(1), np.array([0.5]), step_time=0.5),
    filter=LowPassFilter(cutoff_hz=3.0) % 0.01,
)
print(loop)
loop.sample_period

The loop's state stacks the plant, the filter and the command held between the
controller's firings.

In [ ]:
loop.state.labels

## 3. Two runners: dense and ticks

A sampled loop runs two ways:

- `compute_trajectory(tf, dt=...)` keeps the plant's integration substeps: a dense run in seconds, for plots;
- `compute_rollout(n_steps)` keeps the ticks only: the discrete-time view, the one planners and evaluators use.

Both run the same scan. At each tick the two agree exactly.

In [ ]:
dense = loop.compute_trajectory(tf=3.0, dt=0.002)
tick_run = loop.compute_rollout(n_steps=300)

substeps_per_tick = 5
(
    dense.t.shape,
    tick_run.t.shape,
    np.allclose(dense.x[:, ::substeps_per_tick], tick_run.x),
)

In [ ]:
loop.plot_trajectory(dense)

## 4. Discrete-time analysis

`linearize` of a sampled loop returns a `DiscreteLTISystem`,
$x_{k+1} = A_d x_k + B_d u_k$, with its period $\Delta$. Its poles live in the $z$-plane:
the loop is stable when every pole is inside the unit circle, $|z| < 1$.

A PID that holds a pendulum at rest becomes unstable when its computer is too slow. The
loop is linearized at the hanging rest for four periods:

In [ ]:
from minilink import PID, Pendulum
from minilink.analysis import linearize

pendulum = Pendulum()
pid = PID(Kp=20.0, Ki=5.0, Kd=4.0)

for dt in (0.01, 0.1, 0.2, 0.4):
    digital_loop = (pid % dt) @ pendulum
    model = linearize(digital_loop, x_bar=np.zeros(digital_loop.n))
    radius = np.abs(np.linalg.eigvals(model.params["A"])).max()
    print(f"dt = {dt:4.2f} s   max |z| = {radius:.4f}")

The radius crosses 1 at $\Delta t = 0.2$ s, and not because of the plant: the PID's
derivative filter, $\tau \dot z = e - z$ with $\tau = 0.1$ s, is a continuous law, and its
Euler step has the pole $z = 1 - \Delta t / \tau$, which leaves the unit circle once
$\Delta t > 2\tau$. A digital law needs its time constants slower than its clock.

The model is a system like any other. Printed, it shows its matrices and its period; the
pole-zero map draws the unit circle with the poles of the slowest computer, one at $z = -3$.

In [ ]:
from minilink.analysis import plot_pzmap

print(model)
plot_pzmap(model)

The frequency response of a discrete model is $G(e^{j\omega\Delta})$: it is periodic, and
the Bode diagram stops at the Nyquist frequency $\pi / \Delta$. `plot_bode` from
`minilink.analysis` takes the sampled loop directly; here the closed loop at 20 Hz, from the
reference to the angle.

In [ ]:
from minilink.analysis import plot_bode

loop_20hz = (pid % 0.05) @ pendulum
plot_bode(loop_20hz, x_bar=np.zeros(loop_20hz.n), margins=False)

## Summary

| Object | Time axis | Built by | Runs with |
| --- | --- | --- | --- |
| `StepSystem`, `StepDiagramSystem` | ticks $k$ | subclass, `@`, `>>` | `compute_rollout` |
| `SampledSystem`, `SampledDiagramSystem` of laws | ticks of $\Delta t$ seconds | `law % dt`, `diagram % dt` | `compute_rollout` |
| sampled loop on a plant | ticks + continuous plant | `(law % dt) @ plant`, `closed_loop(..., filter=...)` | `compute_rollout`, `compute_trajectory` |
| `DiscreteLTISystem` | ticks of $\Delta$ | `linearize(sampled_loop)` | poles in $z$, `plot_pzmap`, `plot_bode` |

Next, [`02_optimal_control`](02_optimal_control.ipynb): one problem, four planners.